# RAG Demo — Knowledge-Grounded Answers with LlamaStack

This notebook demonstrates RAG (Retrieval-Augmented Generation) using LlamaStack's
Agent API with the `file_search` tool against pre-ingested documents in pgvector.

### Vector Stores Available

| Store | Documents | Use Case |
|-------|-----------|----------|
| `acme_corporate` | 8 lithography/manufacturing docs | ACME demo scenario |
| `whoami` | Your CV | Personal demo |

### Pattern
Inspired by [burrsutter/fantaco-redhat-one-2026](https://github.com/burrsutter/fantaco-redhat-one-2026/tree/main/rag-llama-stack)

In [ ]:
!pip install -q llama-stack-client python-dotenv rich

In [ ]:
import os, logging
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("llama_stack_client").setLevel(logging.WARNING)

from llama_stack_client import LlamaStackClient

LLAMA_STACK_URL = os.getenv("LLAMA_STACK_URL", "http://lsd-rag-service.private-ai.svc.cluster.local:8321")
INFERENCE_MODEL = os.getenv("INFERENCE_MODEL", "vllm-inference/qwen3-8b-agent")

client = LlamaStackClient(base_url=LLAMA_STACK_URL)

print(f"LlamaStack: {LLAMA_STACK_URL}")
print(f"Model:      {INFERENCE_MODEL}")
print(f"\nModels available:")
for m in client.models.list():
    print(f"  {m.identifier}")

---
## 1. List Vector Stores

In [ ]:
stores = list(client.vector_stores.list())
print(f"Vector stores: {len(stores)}\n")
store_map = {}
for vs in stores:
    fc = vs.file_counts
    done = fc.completed if hasattr(fc, 'completed') else 0
    print(f"  {vs.name:20s}  id={vs.id}  files={done}")
    store_map[vs.name] = vs.id

---
## 2. Direct Vector Search (Debug)

Test retrieval quality before running the Agent.

In [ ]:
def search_store(store_name, query, max_results=3):
    vs_id = store_map.get(store_name)
    if not vs_id:
        print(f"Store '{store_name}' not found")
        return
    results = client.vector_stores.search(
        vector_store_id=vs_id,
        query=query,
        max_num_results=max_results,
    )
    chunks = results.data if hasattr(results, 'data') else []
    print(f"Store: {store_name} | Query: '{query}' | Chunks: {len(chunks)}")
    for i, r in enumerate(chunks):
        text = r.content[0].text[:150] if hasattr(r, 'content') and r.content else str(r)[:150]
        score = r.score if hasattr(r, 'score') else 0
        print(f"  [{i+1}] score={score:.3f}: {text}...")
    print()

In [ ]:
search_store("acme_corporate", "L-900 DFO calibration drift procedure")
search_store("whoami", "Who is Adnan Drina and what is his experience?")

---
## 3. Agent-Based RAG Queries

The Agent uses `file_search` tool to retrieve context from pgvector,
then generates grounded answers.

In [ ]:
from llama_stack_client import Agent, AgentEventLogger

def ask_with_rag(store_name, question, instructions=None):
    vs_id = store_map.get(store_name)
    if not vs_id:
        print(f"Store '{store_name}' not found")
        return

    if not instructions:
        instructions = (
            "You MUST use the file_search tool to answer ALL questions. "
            "Base your answer strictly on the retrieved documents. "
            "If the answer is not in the documents, say so clearly."
        )

    agent = Agent(
        client,
        model=INFERENCE_MODEL,
        instructions=instructions,
        tools=[{
            "type": "file_search",
            "vector_store_ids": [vs_id],
        }],
    )

    session_id = agent.create_session(f"rag-{store_name}")
    print(f"Store: {store_name}")
    print(f"Q: {question}")
    print(f"A: ", end="")

    response = agent.create_turn(
        messages=[{"role": "user", "content": question}],
        session_id=session_id,
        stream=True,
    )

    for log in AgentEventLogger().log(response):
        print(log, end="")
    print("\n")

In [ ]:
ask_with_rag("whoami", "Who is Adnan Drina? What is his current role and key expertise?")

In [ ]:
ask_with_rag("acme_corporate", "What is the recommended procedure for L-900 DFO calibration drift?")

In [ ]:
ask_with_rag("acme_corporate", "What products and services does ACME Lithography Systems provide?")

In [ ]:
ask_with_rag("whoami", "What events or conferences has Adnan Drina spoken at?")

---
## 4. Comparison: With vs Without RAG

Show the model answering the same question with and without document context.

In [ ]:
from rich import print as rprint
from rich.table import Table
from rich.panel import Panel

question = "Who is Adnan Drina?"

# Without RAG
print("Querying WITHOUT RAG context...")
no_rag_agent = Agent(
    client,
    model=INFERENCE_MODEL,
    instructions="Answer concisely.",
)
no_rag_session = no_rag_agent.create_session("no-rag")
no_rag_response = no_rag_agent.create_turn(
    messages=[{"role": "user", "content": question}],
    session_id=no_rag_session,
)
no_rag_text = ""
for event in no_rag_response:
    if hasattr(event, 'event') and hasattr(event.event, 'payload'):
        payload = event.event.payload
        if hasattr(payload, 'text'):
            no_rag_text += payload.text

# With RAG
print("Querying WITH RAG context (whoami store)...")
rag_agent = Agent(
    client,
    model=INFERENCE_MODEL,
    instructions="You MUST use the file_search tool. Answer based on documents only.",
    tools=[{"type": "file_search", "vector_store_ids": [store_map["whoami"]]}],
)
rag_session = rag_agent.create_session("with-rag")
rag_response = rag_agent.create_turn(
    messages=[{"role": "user", "content": question}],
    session_id=rag_session,
)
rag_text = ""
for event in rag_response:
    if hasattr(event, 'event') and hasattr(event.event, 'payload'):
        payload = event.event.payload
        if hasattr(payload, 'text'):
            rag_text += payload.text

# Display comparison
table = Table(title=f"RAG Comparison: '{question}'", show_lines=True)
table.add_column("Without RAG", style="yellow", no_wrap=False, width=45)
table.add_column("With RAG (whoami)", style="green", no_wrap=False, width=45)
table.add_row(
    no_rag_text[:500] or "(no response)",
    rag_text[:500] or "(no response)",
)
rprint(table)

---
## 5. Upload New Document (Optional)

Upload a new PDF and add it to a vector store.

In [ ]:
# Uncomment to upload a new document
# import io
# 
# STORE_NAME = "whoami"  # or create a new store
# FILE_PATH = "/path/to/your/document.pdf"
# 
# with open(FILE_PATH, "rb") as f:
#     uploaded = client.files.create(file=f, purpose="assistants")
#     print(f"Uploaded: {uploaded.id}")
# 
# result = client.vector_stores.files.create(
#     vector_store_id=store_map[STORE_NAME],
#     file_id=uploaded.id,
#     chunking_strategy={
#         "type": "static",
#         "static": {"max_chunk_size_tokens": 700, "chunk_overlap_tokens": 100}
#     },
# )
# print(f"Ingested: status={result.status}")

---
## References

- [RHOAI 3.3 — Working with LlamaStack RAG](https://docs.redhat.com/en/documentation/red_hat_openshift_ai_self-managed/3.3/html/working_with_llama_stack/)
- [fantaco-redhat-one-2026 RAG demo](https://github.com/burrsutter/fantaco-redhat-one-2026/tree/main/rag-llama-stack)
- [LlamaStack Client SDK](https://github.com/meta-llama/llama-stack-client-python)